# Assignment 2

Name: Vivek Mule
Roll: 381072
PRN: 22420145

To perform data pre-processing and partitioning for a Federated Learning system using a real 
dataset such as a student performance dataset (e.g., study hours, attendance, internal marks, final 
score). The dataset is cleaned by handling missing values and normalization, then split into 
multiple local datasets and distributed among participating devices or nodes so that each client 
trains on its own private data without sharing raw information. 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# --- Load dataset ---
data_path = Path("studyhours.csv")
df = pd.read_csv(data_path)

# --- Basic cleaning: drop duplicates, handle missing ---
df = df.drop_duplicates().copy()
if df.isna().any().any():
    # For simplicity, drop rows with any missing values
    df = df.dropna().copy()

# Ensure correct dtypes
num_cols = ["study_hours", "attendance", "internal_marks", "final_score"]
df[num_cols] = df[num_cols].astype(float)

# --- Normalize features (fit on full data for demo) ---
scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[num_cols])
df_scaled = pd.DataFrame(scaled_values, columns=num_cols)

# --- Partition into federated clients ---

def partition_clients(df_features: pd.DataFrame, num_clients: int = 3, shuffle: bool = True):
    n = len(df_features)
    indices = np.arange(n)
    if shuffle:
        np.random.shuffle(indices)
    splits = np.array_split(indices, num_clients)
    clients = []
    for cid, idx in enumerate(splits):
        clients.append({
            "client_id": cid,
            "data": df_features.iloc[idx].reset_index(drop=True)
        })
    return clients

clients = partition_clients(df_scaled, num_clients=3)

print(f"Total samples: {len(df_scaled)}")
for c in clients:
    print(f"Client {c['client_id']} samples: {len(c['data'])}")
    display(c['data'].head())

Total samples: 4
Client 0 samples: 2


,study_hours,attendance,internal_marks,final_score
0,-1.341641,-1.521278,-1.526564,-1.264911
1,1.341641,1.183216,1.128330,1.264911


Client 1 samples: 1


,study_hours,attendance,internal_marks,final_score
0,0.447214,0.507093,0.597351,0.632456


Client 2 samples: 1


,study_hours,attendance,internal_marks,final_score
0,-0.447214,-0.169031,-0.199117,-0.632456
